# Definitions

Let subscripts denote nodal or stencil and superscripts denote edge quantites. Let

- $\mathbf{e}^i=[x_{i+1}, y_{i+1}, z_{i+1}]^T - [x_i, y_i, z_i]^T$ be an edge.
- $\mathbf{t}^i=\mathbf{e}^i/\|\mathbf{e}^i\|$ be the tangent vector.
- $\{\mathbf{d}_1^i, \mathbf{d}_2^i, \mathbf{t}^i\}$ be the orthonormal reference frame.
- $\{\mathbf{m}_1^i, \mathbf{m}_2^i, \mathbf{t}^i\}$ be the orthonormal material frame and the rotation of reference frame by a signed angle $\theta^i$ along $\mathbf{t}^i$.

For functions, let

- $P(\mathbf{u}, \mathbf{t}_0, \mathbf{t}_1)$ be the **parallel transport** of $\mathbf{u}$ rotated exactly as $\mathbf{t}_0$ rotates to $\mathbf{t}_1$.
- $R(\mathbf{u}, \mathbf{n}, \theta)$ be $\mathbf{u}$ rotated around the axis $\mathbf{n}$ by angle $\theta$.
- $\angle(\mathbf{a}, \mathbf{b}, \mathbf{n})$ be the **signed angle** from $\mathbf{a}$ to $\mathbf{b}$ measured around the normal axis $\mathbf{n}$.

# Formulation

For a 11 node stencil, let

- $\mathbf{q}=[x_0, y_0, z_0, \theta^0, x_1, y_1, z_1, \theta^1, x_2, y_2, z_2]^T$ be the state vector.
- $\mathbf{aux}=[\mathbf{t}^0_{\textrm{old}}, \mathbf{t}^1_{\textrm{old}}, \mathbf{d}_{1,\textrm{old}}^0, \mathbf{d}_{1,\textrm{old}}^1, \beta_\textrm{old}]^T$ be the auxiliary vector.
- $\boldsymbol{\epsilon}=[\epsilon^0, \epsilon^1, \kappa_1, \kappa_2, \tau]^T$ be the strain vector.
- $E=f(\boldsymbol{\epsilon})$ be the energy.

### Update $\mathbf{q}$ and $\mathbf{aux}$

After calculating $\Delta\mathbf{q}$ from Newton-Raphson or similiar methods, we must update both $\mathbf{q}$ and $\mathbf{aux}$:

- $\mathbf{q}_\textrm{new}=\mathbf{q}_\textrm{old}+\Delta{q}$
- $\mathbf{t}^i_\textrm{new}=\mathbf{e}^i_\textrm{new}/\|\mathbf{e}^i_\textrm{new}\|$
- $\mathbf{d}^i_{1,\textrm{new}}=\textrm{P}(\mathbf{d}^i_{1,\textrm{old}}, \mathbf{t}^i_\textrm{old}, \mathbf{t}^i_\textrm{new})$
- $\beta_{\text{new}} = \beta_{\text{old}} + \angle \Big( R \big( P(\mathbf{d}^0_{1,\text{new}}, \mathbf{t}^0_{\text{new}}, \mathbf{t}^1_{\text{new}}), \mathbf{t}^1_{\text{new}}, \beta_{\text{old}} \big), \mathbf{d}^1_{1,\text{new}}, \mathbf{t}^1_{\text{new}} \Big)$

## Frame Reconstruction

First, we calculate $\mathbf{d}_{1,\textrm{new}}^i$ from $\mathbf{aux}$ and $\mathbf{q}$

$$
\mathbf{d}_{1,\textrm{new}}^i=\textrm{P}(\mathbf{d}_{1,\textrm{old}}^i,\mathbf{t}_{\textrm{old}}^i, \mathbf{t}_{\textrm{new}}^i)
$$

With $\mathbf{t}_{\textrm{new}}^i$ and $\mathbf{d}_{1,\textrm{new}}^i$ we construct the **new** orthonormal reference frame $\{\mathbf{d}_1^i, \mathbf{d}_2^i, \mathbf{t}^i\}$ as

$$
\mathbf{d}_{2,\textrm{new}}^i=\mathbf{t}^i_{\textrm{new}}\times \mathbf{d}_{1,\textrm{new}}^i
$$

With $\{\mathbf{d}_1^i, \mathbf{d}_2^i, \mathbf{t}^i\}$ and $\theta^i$, we compute the material frame $\{\mathbf{m}_1^i, \mathbf{m}_2^i, \mathbf{t}^i\}$

$$
\begin{align*}
m_1^i&=\cos(\theta^i)\mathbf{d}_1^i+\sin(\theta^i)\mathbf{d}_2^i \\
m_2^i&=-\sin(\theta^i)\mathbf{d}_1^i+\cos(\theta^i)\mathbf{d}_2^i
\end{align*}
$$

## Strains

### Stretching Strain 

The scalar stretching strain is the ratio of elongation / compression of an edge:

$$
\epsilon^i=\frac{\|\mathbf{e}^i\|}{\|\mathbf{\bar e}^i\|}-1
$$

#### Bending Strains

The discrete curvature binormal $(\kappa\mathbf{b})$ at node $i$ represents the integrated curvature between edges $\mathbf{e}^0$ and $\mathbf{e}^1$:

$$
(\kappa\mathbf{b})=\frac{2(\mathbf{t^0}\times \mathbf{t^1})}{1+\mathbf{t^0}\cdot \mathbf{t^1}}
$$

The scalar bending strains are the projections of this binormal onto the averaged material directors:

$$
\begin{align*}
\kappa_1&=\frac{1}{2}(\kappa\mathbf{b})\cdot(\mathbf{m}_2^0+\mathbf{m}_2^1) \\
\kappa_2&=-\frac{1}{2}(\kappa\mathbf{b})\cdot(\mathbf{m}_1^0+\mathbf{m}_1^1) 
\end{align*}
$$

### Twisting Strain

The twisting strain $\tau$ accounts for the relative rotation of material frames plus the geometric holonomy:

$$
\tau=\theta^1-\theta^0+\beta
$$

## Energy

$$
E=f(\boldsymbol{\epsilon})
$$


In [ ]:
@wp.kernel
def hess_kappa_der(
    nodes: wp.array(dtype=wp.vec3),
    m1s: wp.array(dtype=wp.vec3),
    m2s: wp.array(dtype=wp.vec3),
    ks: wp.array(dtype=float),
    F: wp.array(dtype=float),
):
    idx = wp.tid()
    n0, n1, n2 = nodes[idx], nodes[idx + 1], nodes[idx + 2]
    m1e, m2e = m1s[idx * 2], m2s[idx * 2]
    m1f, m2f = m1s[idx * 2 + 1], m2s[idx * 2 + 1]

    ee = n1 - n0
    ef = n2 - n1
    inv_ee = 1.0 / wp.length(ee)
    inv_ef = 1.0 / wp.length(ef)
    te = ee * inv_ee
    tf = ef * inv_ef

    inv_denom = 1.0 / (1.0 + wp.dot(te, tf))  # TODO: add eps?
    kb = 2.0 * wp.cross(te, tf) * inv_denom

    kappa1 = 0.5 * wp.dot(kb, m2e + m2f)
    kappa2 = -0.5 * wp.dot(kb, m1e + m1f)

    tilde_t = (te + tf) * inv_denom
    # tilde_d1 = (m1e + m1f) * inv_denom
    tilde_d2 = (m2e + m2f) * inv_denom

    tt_o_tt = wp.outer(tilde_t, tilde_t)
    tf_c_d2t_o_tt = wp.outer(wp.cross(tf, tilde_d2), tilde_t)
    tt_o_tf_c_d2t = wp.transpose(tf_c_d2t_o_tt)
    kb_o_d2e = wp.outer(kb, m2e)
    d2e_o_kb = wp.transpose(kb_o_d2e)

    I3 = wp.mat33(1.0, 0.0, 0.0, 0.0, 1.0, 0.0, 0.0, 0.0, 1.0)

    D2kappa1De2 = (
        inv_ee * inv_ee * (2.0 * kappa1 * tt_o_tt - tf_c_d2t_o_tt - tt_o_tf_c_d2t)
        - kappa1 * inv_denom * inv_ee * inv_ee * (I3 - wp.outer(te, te))
        + 0.25 * inv_ee * inv_ee * (kb_o_d2e + d2e_o_kb)
    )

    te_c_d2t_o_tt = wp.outer(wp.cross(te, tilde_d2), tilde_t)
    tt_o_te_c_d2t = wp.transpose(te_c_d2t_o_tt)
    kb_o_d2f = wp.outer(kb, m2f)
    d2f_o_kb = wp.transpose(kb_o_d2f)
    tf_o_tf = wp.outer(tf, tf)

    D2kappa1Df2 = (
        inv_ef * inv_ef * (2.0 * kappa1 * tt_o_tt + te_c_d2t_o_tt + tt_o_te_c_d2t)
        - kappa1 * inv_denom * inv_ee * inv_ee * (I3 - tf_o_tf)
        + 0.25 * inv_ef * inv_ef * (kb_o_d2f + d2f_o_kb)
    )

    D2kappa1DeDf = -kappa1 * inv_denom * inv_ee * inv_ef * (I3 + wp.outer(te, tf))
    +1.0 * inv_ee * inv_ef * (
        2.0 * kappa1 * tt_o_tt - tf_c_d2t_o_tt + tt_o_te_c_d2t
        #    - crossMat(tilde_d2)  # todo cross mat
    )
    D2kappa1DfDe = wp.transpose(D2kappa1DeDf)

    D2kappa1Dthetae = -0.5 * wp.dot(kb, m2e)
    D2kappa1Dthetaf2 = -0.5 * wp.dot(kb, m2f)
    D2kappa2Dthetae2 = 0.5 * wp.dot(kb, m1e)
    D2kappa2Dthetaf2 = 0.5 * wp.dot(kb, m1f)

    D2kappa1DeDthetae = inv_ee * (
        0.5 * wp.dot(kb, m1e) * tilde_t - inv_denom * wp.cross(tf, m1e)
    )
    D2kappa1DeDthetaf = inv_ee * (
        0.5 * wp.dot(kb, m1f) * tilde_t - inv_denom * wp.cross(tf, m1f)
    )
    D2kappa1DfDthetae = inv_ef * (
        0.5 * wp.dot(kb, m1e) * tilde_t + inv_denom * wp.cross(te, m1e)
    )
    D2kappa1DfDthetaf = inv_ef * (
        0.5 * wp.dot(kb, m1f) * tilde_t + inv_denom * wp.cross(te, m1f)
    )
    D2kappa2DeDthetae = inv_ee * (
        0.5 * wp.dot(kb, m2e) * tilde_t - inv_denom * wp.cross(tf, m2e)
    )
    D2kappa2DeDthetaf = inv_ee * (
        0.5 * wp.dot(kb, m2f) * tilde_t - inv_denom * wp.cross(tf, m2f)
    )
    D2kappa2DfDthetae = inv_ef * (
        0.5 * wp.dot(kb, m2e) * tilde_t + inv_denom * wp.cross(te, m2e)
    )
    D2kappa2DfDthetaf = inv_ef * (
        0.5 * wp.dot(kb, m2f) * tilde_t + inv_denom * wp.cross(te, m2f)
    )
